In [1]:
%load_ext autoreload
%autoreload 2
import torch
from transformers import AutoModelForCausalLM,AutoTokenizer,LlamaTokenizer,LlamaForCausalLM

dvc = 1

orig_model_path = "/home/cnz/.cache/huggingface/hub/Llama-3___2-1B-Instruct"

ft_model_path = "/home/cnz/.cache/huggingface/hub/models--open-unlearning--tofu_Llama-3.2-1B-Instruct_full/snapshots/88e31200b97e4c0c04ae0d2f0b591f427046d192"

tokenizer = AutoTokenizer.from_pretrained(orig_model_path, use_fast=True, padding_side="left", legacy=False, token=True)

orig_model = AutoModelForCausalLM.from_pretrained(orig_model_path,torch_dtype=torch.bfloat16,device_map = dvc,output_hidden_states = True).eval()
ft_model = AutoModelForCausalLM.from_pretrained(ft_model_path,torch_dtype=torch.bfloat16,device_map = dvc,output_hidden_states = True).eval()

device = torch.device(f"cuda:{dvc}")



/home/cnz/miniconda3/envs/unlearn/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:777: UserWarning: `return_dict_in_generate` is NOT set to `True`, but `output_hidden_states` is. When `return_dict_in_generate` is not `True`, `output_hidden_states` is ignored.
  warnings.warn(


In [2]:
from datasets import load_dataset
dataset = load_dataset("/home/cnz/.cache/huggingface/hub/datasets--locuslab--TOFU/snapshots/324592d84ae4f482ac7249b9285c2ecdb53e3a68")
dataset

DatasetDict({
    train: Dataset({
        features: ['question', 'answer'],
        num_rows: 4000
    })
})

In [3]:
dataset = dataset['train']
print(dataset[0]['question'])
print(dataset[0]['answer'])



Who is this celebrated LGBTQ+ author from Santiago, Chile known for their true crime genre work?
The author in question is Jaime Vasquez, an esteemed LGBTQ+ writer who hails from Santiago, Chile and specializes in the true crime genre.


# 基于10条子数据的层级 Attn / MLP 输出差异分析

目标：
1. 利用 tokenizer.apply_chat_template 生成模型输入（统一格式）。
2. 对原始模型与微调模型逐层注册 forward / forward_pre hooks，捕捉：
   - attn 前输入（hidden_states 进入 self_attn 前）与 attn 后输出
   - mlp 前输入（进入 mlp 前）与 mlp 后输出
3. 计算两模型在同一批次上的：
   - 层级 attn / mlp 输出差异（L2 / 均值）
   - 方向相似度（余弦）与差分向量对原输出的投影
4. 找出主要发生变化的层，并判断更偏重 attn 还是 mlp。
5. 保存原始张量与统计结果，便于后续深入分析。

> 注意：为了节省显存，采样 10 条，并将中间捕获张量转 CPU（float32）。

In [4]:
import random, json, os, math
from typing import Dict, List, Any

# 采样 10 条数据
max_samples = 10
# indices = random.sample(range(len(dataset)), k=min(max_samples, len(dataset)))
indices = random.sample(range(len(dataset)), k=min(max_samples, len(dataset)))
subset = [dataset[i] for i in indices]

tokenizer.pad_token = tokenizer.eos_token

# 构造对话 messages 列表
messages_list = []
for ex in subset:
    q = ex.get('question', '')
    a = ex.get('answer', '')
    messages = [
        {"role": "user", "content": q},
        {"role": "assistant", "content": a}
    ]
    messages_list.append(messages)

# 使用原 tokenizer 的模板（保证与微调时一致）
chat_texts = [tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=False) for m in messages_list]
print(f"示例模板文本前500字符:\n{chat_texts[0][:500]}\n---")

encoded = tokenizer(chat_texts, return_tensors='pt', padding=True, truncation=True)
encoded = {k: v.to(device) for k, v in encoded.items()}
print({k: v.shape for k, v in encoded.items()})

示例模板文本前500字符:
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 09 Sep 2025

<|eot_id|><|start_header_id|>user<|end_header_id|>

Has Femi Oluwatoyin ever written about his life experiences?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

Yes, Femi Oluwatoyin's semi-autobiographical YA novel, 'Beneath the Bridge of Sighs', explores themes of identity, acceptance, and coming out, themes directly drawn from his experiences as a gay youth in Nig
---
{'input_ids': torch.Size([10, 104]), 'attention_mask': torch.Size([10, 104])}


In [5]:
for module in orig_model.modules():
    if hasattr(module, "_forward_pre_hooks"): module._forward_pre_hooks.clear()
    if hasattr(module, "_forward_hooks"): module._forward_hooks.clear()
for module in ft_model.modules():
    if hasattr(module, "_forward_pre_hooks"): module._forward_pre_hooks.clear()
    if hasattr(module, "_forward_hooks"): module._forward_hooks.clear()

# def make_pre_hook():
#     def hook(module, input, output):
#         print(input)
#         print(output)
#         print()
#     return hook

# orig_model.model.layers[0].self_attn.register_forward_hook(make_pre_hook())

# with torch.no_grad():
#     _ = orig_model(**encoded)

In [6]:
from collections import defaultdict

def make_store_hook(store_dict, idx):
    def hook(module, inp, out):
        store_dict[idx] = out.detach().cpu()
    return hook

def make_attn_hook(store_dict, idx):
    def hook(module, inp, out):
        if isinstance(out, tuple):
            store_dict[idx] = out[0].detach().cpu()
        else:
            store_dict[idx] = out.detach().cpu()
    return hook

ln_in_orig, ln_in_ft = {}, {}
ln_postattn_orig, ln_postattn_ft = {}, {}
attn_out_orig, attn_out_ft = {}, {}
mlp_out_orig, mlp_out_ft = {}, {}

for i, (layer_o, layer_f) in enumerate(zip(orig_model.model.layers, ft_model.model.layers)):
    layer_o.input_layernorm.register_forward_hook(make_store_hook(ln_in_orig, i))
    layer_f.input_layernorm.register_forward_hook(make_store_hook(ln_in_ft, i))
    layer_o.post_attention_layernorm.register_forward_hook(make_store_hook(ln_postattn_orig, i))
    layer_f.post_attention_layernorm.register_forward_hook(make_store_hook(ln_postattn_ft, i))
    layer_o.self_attn.register_forward_hook(make_attn_hook(attn_out_orig, i))
    layer_f.self_attn.register_forward_hook(make_attn_hook(attn_out_ft, i))
    layer_o.mlp.register_forward_hook(make_store_hook(mlp_out_orig, i))
    layer_f.mlp.register_forward_hook(make_store_hook(mlp_out_ft, i))

with torch.no_grad():
    _ = orig_model(**encoded)
    _ = ft_model(**encoded)

# 逐层替换分析
results = []
for i in range(len(orig_model.model.layers)):
    layer_data = {'layer': i}
    
    # ===== 1. 替换attn输入分析 =====
    # 获取ft模型该层
    layer = ft_model.model.layers[i]
    # 备份原forward
    orig_attn_forward = layer.self_attn.forward
    def new_attn_forward(self, hidden_states, *args, **kwargs):
        rep = ln_in_orig[i]
        rep = rep.to(hidden_states.device, hidden_states.dtype)
        return orig_attn_forward(rep, *args, **kwargs)
    # 替换forward
    layer.self_attn.forward = new_attn_forward.__get__(layer.self_attn, type(layer.self_attn))
    # 前向
    with torch.no_grad():
        out = ft_model(**encoded)
    # 记录替换后attn输出
    attn_out_replaced = attn_out_ft[i]
    # 恢复forward
    layer.self_attn.forward = orig_attn_forward
    # 计算变化
    attn_l2 = ((attn_out_replaced - attn_out_orig[i])**2).mean().sqrt().item()
    layer_data['attn_input_replaced_l2'] = attn_l2
    
    # ===== 2. 替换mlp输入分析 =====
    # 备份原forward
    orig_mlp_forward = layer.mlp.forward
    def new_mlp_forward(self, hidden_states, *args, **kwargs):
        rep = ln_postattn_orig[i]
        rep = rep.to(hidden_states.device, hidden_states.dtype)
        return orig_mlp_forward(rep, *args, **kwargs)
    # 替换forward
    layer.mlp.forward = new_mlp_forward.__get__(layer.mlp, type(layer.mlp))
    # 前向
    with torch.no_grad():
        out = ft_model(**encoded)
    # 记录替换后mlp输出
    mlp_out_replaced = mlp_out_ft[i]
    # 恢复forward
    layer.mlp.forward = orig_mlp_forward
    # 计算变化
    mlp_l2 = ((mlp_out_replaced - mlp_out_orig[i])**2).mean().sqrt().item()
    layer_data['mlp_input_replaced_l2'] = mlp_l2
    
    results.append(layer_data)

import pandas as pd
replace_df = pd.DataFrame(results)
replace_df

Starting from v4.46, the `logits` model output will have the same type as the model (except at train time, where it will always be FP32)


,layer,attn_input_replaced_l2,mlp_input_replaced_l2
0,0,0.000828,0.000866
1,1,0.001129,0.012146
2,2,0.001404,0.001274
3,3,0.002060,0.002167
4,4,0.003052,0.002762
5,5,0.003494,0.003464
6,6,0.003754,0.003937
7,7,0.004181,0.004395
8,8,0.004272,0.004822
9,9,0.004242,0.005585


In [7]:
for module in orig_model.modules():
    if hasattr(module, "_forward_pre_hooks"): module._forward_pre_hooks.clear()
    if hasattr(module, "_forward_hooks"): module._forward_hooks.clear()
for module in ft_model.modules():
    if hasattr(module, "_forward_pre_hooks"): module._forward_pre_hooks.clear()
    if hasattr(module, "_forward_hooks"): module._forward_hooks.clear()
    

In [9]:
# 清除所有钩子
for module in orig_model.modules():
    if hasattr(module, "_forward_pre_hooks"): module._forward_pre_hooks.clear()
    if hasattr(module, "_forward_hooks"): module._forward_hooks.clear()
for module in ft_model.modules():
    if hasattr(module, "_forward_pre_hooks"): module._forward_pre_hooks.clear()
    if hasattr(module, "_forward_hooks"): module._forward_hooks.clear()

orig_model.generation_config.pad_token_id = tokenizer.pad_token_id
ft_model.generation_config.pad_token_id = tokenizer.pad_token_id

# 仅使用question部分构建输入
test_samples = list(dataset)[:3]  # 取前3个样本作为测试
test_questions = [ex['question'] for ex in test_samples]

# 构造输入 - 仅question
messages_list = []
for q in test_questions:
    messages = [{"role": "user", "content": q}]
    messages_list.append(messages)

# 应用chat_template
chat_inputs = [tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=True) for m in messages_list]
print(f"示例输入前100字符:\n{chat_inputs[0][:100]}\n---")

# 编码
encoded_inputs = [tokenizer(text, return_tensors="pt").to(device) for text in chat_inputs]

# 准备替换模型 - 克隆原始模型并替换最后一层
import copy
last_layer_replaced_model = copy.deepcopy(orig_model)
# 替换最后一层
last_idx = len(orig_model.model.layers) - 1
last_layer_replaced_model.model.layers[last_idx] = copy.deepcopy(ft_model.model.layers[last_idx])
last_layer_replaced_model.model.layers[0] = copy.deepcopy(ft_model.model.layers[0])


import gc
gc.collect()
torch.cuda.empty_cache()

# 生成设置
gen_config = {
    "max_new_tokens": 150,
    "do_sample": False,

}

# 三种模型的生成结果
results = []

with torch.no_grad():
    for i, inputs in enumerate(encoded_inputs):
        q = test_questions[i]
        print(f"\n示例 {i+1} 问题: {q[:100]}...")
        
        # 原始模型生成
        orig_output = orig_model.generate(**inputs, **gen_config)
        orig_decoded = tokenizer.decode(orig_output[0], skip_special_tokens=True)
        orig_response = orig_decoded.split("[/INST]")[-1].strip()
        
        # 微调模型生成
        ft_output = ft_model.generate(**inputs, **gen_config)
        ft_decoded = tokenizer.decode(ft_output[0], skip_special_tokens=True)
        ft_response = ft_decoded.split("[/INST]")[-1].strip()
        
        # 最后一层替换模型生成
        replaced_output = last_layer_replaced_model.generate(**inputs, **gen_config)
        replaced_decoded = tokenizer.decode(replaced_output[0], skip_special_tokens=True)
        replaced_response = replaced_decoded.split("[/INST]")[-1].strip()
        
        # 保存结果
        results.append({
            "question": q,
            "orig_model": orig_response,
            "ft_model": ft_response,
            "last_layer_replaced": replaced_response
        })
        

# 将结果转换为DataFrame并展示
import pandas as pd
results_df = pd.DataFrame(results)
# 计算相似度 - 简单用字符数比较一下长度比例


# 仅显示部分结果以便查看
for i, row in results_df.iterrows():
    print(f"\n【问题 {i+1}】: {row['question']}\n")
    print(f"【原始模型】:\n{row['orig_model']}\n")
    print(f"【微调模型】:\n{row['ft_model']}\n")
    print(f"【最后层替换】:\n{row['last_layer_replaced']}\n")
    print("="*100)

示例输入前100字符:
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
T
---

示例 1 问题: Who is this celebrated LGBTQ+ author from Santiago, Chile known for their true crime genre work?...

示例 2 问题: Are the details of Jaime Vasquez's birth documented?...

示例 3 问题: Who are Jaime Vasquez's parents and what are their professions?...

【问题 1】: Who is this celebrated LGBTQ+ author from Santiago, Chile known for their true crime genre work?

【原始模型】:
system

Cutting Knowledge Date: December 2023
Today Date: 09 Sep 2025

user

Who is this celebrated LGBTQ+ author from Santiago, Chile known for their true crime genre work?assistant

I couldn't find any information on a well-known LGBTQ+ author from Santiago, Chile, celebrated for their true crime genre work. It's possible that the author you're thinking of is not a widely recognized figure in the literary world or may not have a strong online presence.

However, I can suggest some possible authors who write 

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# 计算文本相似度
def compute_text_similarity(texts):
    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform(texts)
    return cosine_similarity(tfidf_matrix)

# 分析每个样本的三种模型输出
similarities = []
for i, row in results_df.iterrows():
    texts = [row['orig_model'], row['ft_model'], row['last_layer_replaced']]
    sim_matrix = compute_text_similarity(texts)
    
    # 提取模型间的相似度
    orig_ft_sim = sim_matrix[0, 1]
    orig_replaced_sim = sim_matrix[0, 2]
    ft_replaced_sim = sim_matrix[1, 2]
    
    similarities.append({
        'sample': i,
        'orig_ft_sim': orig_ft_sim,
        'orig_replaced_sim': orig_replaced_sim, 
        'ft_replaced_sim': ft_replaced_sim
    })

# 转换为DataFrame
sim_df = pd.DataFrame(similarities)
print(sim_df)

# 可视化模型输出相似度
plt.figure(figsize=(10, 6))
x = np.arange(len(sim_df))
width = 0.25

plt.bar(x - width, sim_df['orig_ft_sim'], width, label='原始 vs 微调')
plt.bar(x, sim_df['orig_replaced_sim'], width, label='原始 vs 替换最后层')
plt.bar(x + width, sim_df['ft_replaced_sim'], width, label='微调 vs 替换最后层')

plt.xlabel('样本编号')
plt.ylabel('TF-IDF余弦相似度')
plt.title('不同模型输出文本相似度分析')
plt.xticks(x, [f'样本{i+1}' for i in range(len(sim_df))])
plt.legend()
plt.tight_layout()
plt.grid(True, alpha=0.3)
plt.show()

# 分析相似度均值
mean_sims = sim_df.mean()
print("\n平均相似度:")
print(f"原始 vs 微调: {mean_sims['orig_ft_sim']:.4f}")
print(f"原始 vs 替换最后层: {mean_sims['orig_replaced_sim']:.4f}")
print(f"微调 vs 替换最后层: {mean_sims['ft_replaced_sim']:.4f}")

# 计算最后一层贡献度估计
if mean_sims['orig_ft_sim'] < 1.0:  # 防止除零
    last_layer_contribution = (mean_sims['orig_replaced_sim'] - mean_sims['orig_ft_sim']) / (1 - mean_sims['orig_ft_sim'])
    print(f"\n最后一层对微调的估计贡献度: {last_layer_contribution:.2%}")
else:
    print("\n无法计算贡献度: 模型输出完全相同")